# Day 080 — Exercise 2: Building the Trace

**What you'll build:** the functions that turn steps back into text — `format_step`, `format_observation`, and `build_react_prompt`.

**Why it matters:** ReAct keeps a *scratchpad* — a running transcript of everything the agent thought, did, and observed. Each turn you render the latest step and observation back into text and feed the whole scratchpad to the model, so it can reason over its own history.

In [ ]:
import json

def _make_mock_llm(script):
    """Return an llm_fn(messages) that yields each scripted reply in turn.

    Repeats the last reply once the script is exhausted - handy for testing a
    runaway loop (a model that never emits a Final Answer).
    """
    state = {'i': 0}
    def _fn(messages):
        i = state['i']
        state['i'] = min(i + 1, len(script) - 1)
        return script[i]
    return _fn
import ast
import json
import operator

# ── tools reused from Day 79: a safe calculator + a fact-lookup tool ──────────
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
    ast.USub: operator.neg, ast.UAdd: operator.pos,
}


def _eval_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError("unsupported expression")


def safe_calculate(expression):
    """Evaluate arithmetic without eval() (see Day 79)."""
    return _eval_node(ast.parse(expression, mode="eval").body)


_FACTS = {
    "speed of light": "299792458 m/s",
    "pi": "3.14159",
    "earth radius": "6371 km",
    "days in a year": "365",
}


def _lookup(args):
    query = str(args.get("query", "")).lower().strip()
    for key, value in _FACTS.items():
        if query and (query in key or key in query):
            return value
    return "No result found for " + repr(args.get("query", ""))


DEFAULT_TOOLS = {
    "calculator": {
        "description": "Evaluate an arithmetic expression, e.g. 2 * (3 + 4).",
        "parameters": {"expression": "string - the arithmetic to evaluate"},
        "fn": lambda args: str(safe_calculate(args["expression"])),
    },
    "lookup": {
        "description": "Look up a known fact: speed of light, pi, earth radius, "
                       "days in a year.",
        "parameters": {"query": "string - what to look up"},
        "fn": _lookup,
    },
}


def build_tool_descriptions(tools):
    """Render a tool registry as prompt text (Day 79)."""
    lines = []
    for name, spec in tools.items():
        params = ", ".join(spec.get("parameters", {}))
        lines.append("- " + name + "(" + params + "): " + spec["description"])
    return "\n".join(lines)


def safe_parse_json(text):
    """Slice first '{' to last '}' and parse. Returns dict|None (Day 79)."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None

# ── parsing the ReAct format ──────────────────────────────────────────────────
def _line_value(text, prefix):
    """Text after the first line starting with prefix (case-insensitive), else ''."""
    for line in text.splitlines():
        if line.strip().lower().startswith(prefix.lower()):
            return line.strip()[len(prefix):].strip()
    return ""


def _after_marker(text, marker):
    """Everything after marker (case-insensitive), or None if absent."""
    idx = text.lower().find(marker.lower())
    if idx == -1:
        return None
    return text[idx + len(marker):].strip()


def parse_react_step(text):
    """Parse one ReAct step. NEVER raises.

    Returns either:
      {"type": "action", "thought": str, "tool": str, "input": dict}
      {"type": "final",  "thought": str, "answer": str}
    A reply with no recognisable Action falls back to a final answer holding
    the raw text - so a malformed step still ends the loop cleanly.
    """
    thought = _line_value(text, "Thought:")
    final = _after_marker(text, "Final Answer:")
    if final is not None:
        return {"type": "final", "thought": thought, "answer": final}
    action = _line_value(text, "Action:")
    if action:
        args = safe_parse_json(_line_value(text, "Input:")) or {}
        return {"type": "action", "thought": thought, "tool": action, "input": args}
    return {"type": "final", "thought": thought, "answer": text.strip()}


## Task

1. `format_step(step)` — render an action step as `Thought: ...` / `Action: ...` / `Input: <json>` (use `json.dumps(step['input'])`).
2. `format_observation(result)` — `'Observation: ' + str(result)`.
3. `build_react_prompt(task, tools, scratchpad)` — a `system` message describing the ReAct format and the tools (`build_tool_descriptions`), and a `user` message with `Task: ...`, then the `scratchpad` if any, ending on `Thought:` to prompt the model. Return `[system, user]`.

## Your Implementation

In [ ]:
def format_step(step):
    """Render an action step back into ReAct text for the scratchpad."""
    raise NotImplementedError

def format_observation(result):
    """Render a tool result as an 'Observation: ...' line."""
    raise NotImplementedError

def build_react_prompt(task, tools, scratchpad):
    """Build the [system, user] messages for one ReAct step."""
    raise NotImplementedError


In [ ]:

# ── formatting the trace (the scratchpad) ─────────────────────────────────────
def format_step(step):
    """Render an action step back into ReAct text for the scratchpad."""
    return ("Thought: " + step["thought"] + "\n"
            + "Action: " + step["tool"] + "\n"
            + "Input: " + json.dumps(step["input"]))


def format_observation(result):
    """Render a tool result as an Observation line."""
    return "Observation: " + str(result)


def build_react_prompt(task, tools, scratchpad):
    """Build the [system, user] messages for one ReAct step."""
    system = "\n".join([
        "You are a reasoning agent. Solve the task step by step using the "
        "ReAct format: reason, act, observe, repeat.",
        "",
        "Available tools:",
        build_tool_descriptions(tools),
        "",
        "On each turn reply in EXACTLY this format:",
        "Thought: <your reasoning about what to do next>",
        "Action: <one tool name from the list above>",
        'Input: {"<param>": "<value>"}',
        "",
        "You will then receive an Observation with the tool's result.",
        "When you can answer, reply instead with:",
        "Thought: <your final reasoning>",
        "Final Answer: <the answer>",
    ])
    user = "Task: " + str(task)
    if scratchpad:
        user = user + "\n\n" + scratchpad.rstrip()
    user = user + "\n\nThought:"
    return [{"role": "system", "content": system},
            {"role": "user", "content": user}]


## Automated checks

In [ ]:

score, total = 0, 5
try:
    step = {'thought': 'add them', 'tool': 'calculator', 'input': {'expression': '2+2'}}
    fs = format_step(step)
    assert 'Thought: add them' in fs and 'Action: calculator' in fs and 'Input:' in fs
    score += 1; print("✅ format_step renders Thought/Action/Input")

    assert '2+2' in fs
    score += 1; print("✅ format_step serialises the input dict")

    ob = format_observation('4')
    assert ob.startswith('Observation:') and '4' in ob
    score += 1; print("✅ format_observation prefixes 'Observation:'")

    msgs = build_react_prompt('add 2 and 2', DEFAULT_TOOLS, '')
    assert msgs[0]['role'] == 'system' and 'calculator' in msgs[0]['content']
    assert 'Thought' in msgs[0]['content'] and 'Final Answer' in msgs[0]['content']
    score += 1; print("✅ build_react_prompt describes the format + tools")

    msgs2 = build_react_prompt('t', DEFAULT_TOOLS, 'Observation: 4')
    assert '4' in msgs2[1]['content']
    score += 1; print("✅ scratchpad is included in the user message")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── formatting the trace (the scratchpad) ─────────────────────────────────────
def format_step(step):
    """Render an action step back into ReAct text for the scratchpad."""
    return ("Thought: " + step["thought"] + "\n"
            + "Action: " + step["tool"] + "\n"
            + "Input: " + json.dumps(step["input"]))


def format_observation(result):
    """Render a tool result as an Observation line."""
    return "Observation: " + str(result)


def build_react_prompt(task, tools, scratchpad):
    """Build the [system, user] messages for one ReAct step."""
    system = "\n".join([
        "You are a reasoning agent. Solve the task step by step using the "
        "ReAct format: reason, act, observe, repeat.",
        "",
        "Available tools:",
        build_tool_descriptions(tools),
        "",
        "On each turn reply in EXACTLY this format:",
        "Thought: <your reasoning about what to do next>",
        "Action: <one tool name from the list above>",
        'Input: {"<param>": "<value>"}',
        "",
        "You will then receive an Observation with the tool's result.",
        "When you can answer, reply instead with:",
        "Thought: <your final reasoning>",
        "Final Answer: <the answer>",
    ])
    user = "Task: " + str(task)
    if scratchpad:
        user = user + "\n\n" + scratchpad.rstrip()
    user = user + "\n\nThought:"
    return [{"role": "system", "content": system},
            {"role": "user", "content": user}]
```

**Why re-serialise the step into text?** The model only understands text. The scratchpad is the agent's memory of this task — writing each step and observation back into ReAct text is how that memory is carried forward.

</details>